In [1]:
# 1. Import Libraries

import pandas as pd
import numpy as np

In [2]:
# 2. Load Data

purchase_behaviour = pd.read_csv('QVI_purchase_behaviour (1).csv')
transactions = pd.read_excel('QVI_transaction_data (1).xlsx')

In [3]:
# 3. Initial Data Checks

print("=== Purchase Behaviour Info ===")
print(purchase_behaviour.info())

print("\n=== Transactions Info ===")
print(transactions.info())

=== Purchase Behaviour Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72637 entries, 0 to 72636
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   LYLTY_CARD_NBR    72637 non-null  int64 
 1   LIFESTAGE         72637 non-null  object
 2   PREMIUM_CUSTOMER  72637 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.7+ MB
None

=== Transactions Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 264836 entries, 0 to 264835
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   DATE            264836 non-null  int64  
 1   STORE_NBR       264836 non-null  int64  
 2   LYLTY_CARD_NBR  264836 non-null  int64  
 3   TXN_ID          264836 non-null  int64  
 4   PROD_NBR        264836 non-null  int64  
 5   PROD_NAME       264836 non-null  object 
 6   PROD_QTY        264836 non-null  int64  
 7   TOT_SALES       264

In [4]:
# Missing values
print("\n=== Missing Values ===")
print(purchase_behaviour.isnull().sum())
print(transactions.isnull().sum())


=== Missing Values ===
LYLTY_CARD_NBR      0
LIFESTAGE           0
PREMIUM_CUSTOMER    0
dtype: int64
DATE              0
STORE_NBR         0
LYLTY_CARD_NBR    0
TXN_ID            0
PROD_NBR          0
PROD_NAME         0
PROD_QTY          0
TOT_SALES         0
dtype: int64


In [5]:
# Summary statistics
print("\n=== Summary Statistics ===")
print(purchase_behaviour.describe(include='all'))
print(transactions.describe())


=== Summary Statistics ===
        LYLTY_CARD_NBR LIFESTAGE PREMIUM_CUSTOMER
count     7.263700e+04     72637            72637
unique             NaN         7                3
top                NaN  RETIREES       Mainstream
freq               NaN     14805            29245
mean      1.361859e+05       NaN              NaN
std       8.989293e+04       NaN              NaN
min       1.000000e+03       NaN              NaN
25%       6.620200e+04       NaN              NaN
50%       1.340400e+05       NaN              NaN
75%       2.033750e+05       NaN              NaN
max       2.373711e+06       NaN              NaN
                DATE     STORE_NBR  LYLTY_CARD_NBR        TXN_ID  \
count  264836.000000  264836.00000    2.648360e+05  2.648360e+05   
mean    43464.036260     135.08011    1.355495e+05  1.351583e+05   
std       105.389282      76.78418    8.057998e+04  7.813303e+04   
min     43282.000000       1.00000    1.000000e+03  1.000000e+00   
25%     43373.000000      70.000

In [13]:
# 4. Data Cleaning

# Convert DATE column (Excel serial to datetime)

if np.issubdtype(transactions['DATE'].dtype, np.number):
    transactions['DATE'] = pd.to_datetime('1899-12-30') + pd.to_timedelta(transactions['DATE'], unit='D')
else:
    transactions['DATE'] = pd.to_datetime(transactions['DATE'], errors='coerce')

# Date fields required conditional transformation due to mixed data formats 
# (Excel serial vs datetime), ensuring robustness in preprocessing

# Remove invalid transactions
transactions = transactions[transactions['TOT_SALES'] > 0]

# Remove extreme outliers (bulk buyers)
transactions = transactions[transactions['PROD_QTY'] <= 10]

In [14]:
# 5. Feature Engineering

# Extract pack size (grams)
transactions['PACK_SIZE'] = transactions['PROD_NAME'].str.extract(r'(\d+)g').astype(float)

# Extract brand name (first word)
transactions['BRAND'] = transactions['PROD_NAME'].str.split().str[0]

In [15]:
# 6. Merge Datasets

data = pd.merge(transactions, purchase_behaviour, on='LYLTY_CARD_NBR', how='left')

In [17]:
# 7. Create Customer Metrics

customer_metrics = data.groupby('LYLTY_CARD_NBR').agg({
    'TOT_SALES': 'sum',
    'PROD_QTY': 'sum',
    'PACK_SIZE': 'mean'
}).reset_index()

customer_metrics.rename(columns={
    'TOT_SALES': 'TOTAL_SPEND',
    'PROD_QTY': 'TOTAL_QTY',
    'PACK_SIZE': 'AVG_PACK_SIZE'
}, inplace=True)

In [18]:
# 8. Segment-Level Metrics

segment_metrics = data.groupby(['LIFESTAGE', 'PREMIUM_CUSTOMER']).agg(
    TOTAL_SALES=('TOT_SALES', 'sum'),
    TOTAL_QTY=('PROD_QTY', 'sum'),
    NUM_CUSTOMERS=('LYLTY_CARD_NBR', 'nunique'),
    AVG_SPEND_PER_CUSTOMER=('TOT_SALES', 'mean')
).reset_index()

print("\n=== Segment Metrics ===")
print(segment_metrics)


=== Segment Metrics ===
                 LIFESTAGE PREMIUM_CUSTOMER  TOTAL_SALES  TOTAL_QTY  \
0   MIDAGE SINGLES/COUPLES           Budget     35514.80       9496   
1   MIDAGE SINGLES/COUPLES       Mainstream     90803.85      22699   
2   MIDAGE SINGLES/COUPLES          Premium     58432.65      15526   
3             NEW FAMILIES           Budget     21928.45       5571   
4             NEW FAMILIES       Mainstream     17013.90       4319   
5             NEW FAMILIES          Premium     11491.10       2957   
6           OLDER FAMILIES           Budget    168363.25      45065   
7           OLDER FAMILIES       Mainstream    103445.55      27756   
8           OLDER FAMILIES          Premium     80658.40      21771   
9    OLDER SINGLES/COUPLES           Budget    136769.80      35220   
10   OLDER SINGLES/COUPLES       Mainstream    133393.80      34997   
11   OLDER SINGLES/COUPLES          Premium    132263.15      33986   
12                RETIREES           Budget    11314

In [ ]:
 # 9. Create Store-Level Metrics

store_metrics = data.groupby(['STORE_NBR', 'YEARMONTH']).agg(
    TOTAL_SALES=('TOT_SALES', 'sum'),
    N_CUSTOMERS=('LYLTY_CARD_NBR', 'nunique'),
    N_TXNS=('TXN_ID', 'nunique')
).reset_index()

# Derived metric
store_metrics['TXNS_PER_CUSTOMER'] = store_metrics['N_TXNS'] / store_metrics['N_CUSTOMERS']

In [ ]:
# 10. Brand Analysis

brand_analysis = data.groupby(['BRAND', 'LIFESTAGE']).agg(
    TOTAL_SALES=('TOT_SALES', 'sum'),
    TOTAL_QTY=('PROD_QTY', 'sum')
).reset_index()

print("\n=== Brand Analysis ===")
print(brand_analysis.head())


=== Brand Analysis ===
    BRAND               LIFESTAGE  TOTAL_SALES  TOTAL_QTY
0  Burger  MIDAGE SINGLES/COUPLES        660.1        287
1  Burger            NEW FAMILIES        161.0         70
2  Burger          OLDER FAMILIES       1566.3        681
3  Burger   OLDER SINGLES/COUPLES       1271.9        553
4  Burger                RETIREES       1131.6        492


In [ ]:
# 11. Pack Size Analysis

pack_size_analysis = data.groupby('PACK_SIZE').agg(
    TOTAL_SALES=('TOT_SALES', 'sum'),
    TOTAL_QTY=('PROD_QTY', 'sum')
).reset_index()

print("\n=== Pack Size Analysis ===")
print(pack_size_analysis.head())


=== Pack Size Analysis ===
   PACK_SIZE  TOTAL_SALES  TOTAL_QTY
0       70.0       6852.0       2855
1       90.0       9676.4       5692
2      110.0     162765.4      42835
3      125.0       5733.0       2730
4      134.0     177655.5      48019
